In [86]:
import json
import pandas as pd
import shopify
import datetime
import requests
import pytz
import os
import binascii
import time

In [87]:
# Read secrets
with open(r'D:\\Study\\THConsultant\\secret.json', 'r') as jfile:
    secrets = json.load(jfile)
shop_name = secrets['shop_name']
token = secrets['token']
api_key = secrets['api_key']
api_password = secrets['api_password']
shopify_store_domain = secrets['shopify_store_domain']
api_version = secrets['api_version']   

In [94]:
# Shopify settings
orders_endpoint = f'https://{shopify_store_domain}/admin/api/{api_version}/orders.json?status=any'
shop_url = "https://%s:%s@%s.myshopify.com/admin" % (api_key, token, shop_name)
shopify.ShopifyResource.set_site(shop_url)
shopify.Session.setup(api_key=api_key, secret=token)

# Function to fetch all orders with pagination using since_id
def fetch_all_orders():
    orders = []
    last_id = None

    while True:
        if last_id:
            new_orders = shopify.Order.find(status='any', limit=250, since_id=last_id)
        else:
            new_orders = shopify.Order.find(status='any', limit=250)

        if not new_orders:
            break

        orders.extend(new_orders)
        last_id = new_orders[-1].id
        time.sleep(0.5)  

    return orders

# Fetch all orders
orders = fetch_all_orders()


* Take customer information (nums of invoice) 
    -> Reference code: Generate Voucher for filtered customer 

- Index : order_number
- Customer: customer(id)
- Products : line_items


In [95]:
info_dict = {}
for inv in orders:
    if inv.customer:
    # Name
        ord_time = datetime.datetime.strptime(inv.created_at, '%Y-%m-%dT%H:%M:%S%z')
        client_name = f'{inv.customer.last_name} {inv.customer.first_name}'
        name = f"S{ord_time.strftime('%d%m%y')}_{inv.order_number}_{client_name}"
    if name not in info_dict:
        info_dict[name] = {}
        info_dict[name]['Order Number'] = inv.order_number
        # Order - Product, Price, Name, Quantity, Order_Date, 
        info_dict[name]['Order Date'] = ord_time.strftime('%m/%d/%Y')
        info_dict[name]['Product'] = []

        for prod in inv.line_items:
            info_dict[name]['Product'].append([prod.name, prod.quantity])

        # Client: Name, Email, Phone, Address
        info_dict[name]['Client name'] = client_name
        if hasattr(inv.customer, 'phone') and inv.customer:
            info_dict[name]['Phone Number'] = inv.customer.phone
        elif hasattr(inv.customer, 'default_address') and inv.customer.default_address:
            info_dict[name]['Phone Number'] = inv.customer.default_address.phone

        info_dict[name]['Email'] = inv.customer.email

        if hasattr(inv.customer, 'default_address') and inv.customer.default_address:
            temp = inv.customer.default_address
            info_dict[name]['Add'] = f'{temp.address1}, {temp.address2}, {temp.city}, {temp.province}'
        # Process - Due_date, Person
    else: 
        continue

In [90]:
data = pd.DataFrame(info_dict)
data

,S260524_1195_Wegman Kyra,S260524_1194_Persaud Sarika,S260524_1193_Pillar Allison,S250524_1192_Ma Selena,S250524_1191_Treidler Anna,S240524_1190_Baxter Alexandra,S230524_1189_Stull Courtney,S230524_1188_Storm Laure,S210524_1187_Herzog Nele,S210524_1186_Kisin Eugenia,...,S310722_1011_Campbell Lundy,S290722_1010_Zuin Marianna,S220722_1009_Vu Hanh,S220622_1008_Kim,S220422_1007_Szolnoki Tibor,S200422_1006_Gulbinski Evelyn,S110422_1005_Szolnoki Tibor,S080422_1004_Xucla Caroline,S310322_1003_Ringel Dagmar,S220222_1002_Polidano Kristina
Order Number,1195,1194,1193,1192,1191,1190,1189,1188,1187,1186,...,1011,1010,1009,1008,1007,1006,1005,1004,1003,1002
Order Date,05/26/2024,05/26/2024,05/26/2024,05/25/2024,05/25/2024,05/24/2024,05/23/2024,05/23/2024,05/21/2024,05/21/2024,...,07/31/2022,07/29/2022,07/22/2022,06/22/2022,04/22/2022,04/20/2022,04/11/2022,04/08/2022,03/31/2022,02/22/2022
Product,"[[Magenta Simple Silk Dress - Olive / S, 1]]",[[Olive Simple Silk Dress - Red Bordeaux / XXL...,"[[Olive Simple Silk Dress - Olive / M, 1]]","[[Silk Scarf - Matcha, 1], [Green Silk Brocade...","[[Grey Simple Silk Dress - Grey / S, 1]]",[[Green Silk Brocade Simple Dress - M / Green ...,[[Green Silk Brocade Simple Dress - M / Green ...,[[Green Silk Brocade Simple Dress - XXL / Gree...,"[[Grey Simple Silk Dress - Grey / M, 1]]","[[Magenta Simple Silk Dress - Olive / L, 1]]",...,"[[Tip, 1], [Blue silk pants with motifs ""Hỷ"" -...","[[Crochet bag - Black / Beige, 1]]","[[Silk shirt - Blue, 1], [Silk shirt - Black /...","[[Kimono robe - Black / Small, 1]]","[[Silk pants - Large, 1]]","[[Tip, 1], [Simple dress - Blue, 1]]","[[Silk shorts (male) - Blue motifs, 1]]","[[Silk pants - Small / Ocre, 1], [Silk pants -...","[[Tip, 1], [Simple dress Custom order - Gray,...","[[Simple dress, 1], [Silk shirt - Ocre, 1]]"
Client name,Wegman Kyra,Persaud Sarika,Pillar Allison,Ma Selena,Treidler Anna,Baxter Alexandra,Stull Courtney,Storm Laure,Herzog Nele,Kisin Eugenia,...,Campbell Lundy,Zuin Marianna,Vu Hanh,Kim,Szolnoki Tibor,Gulbinski Evelyn,Szolnoki Tibor,Xucla Caroline,Ringel Dagmar,Polidano Kristina
Phone Number,None,None,None,None,None,None,None,None,+4915144904857,None,...,None,None,None,None,None,None,None,None,None,+35679972407
Email,miller.kyra@gmail.com,a.sarika.persaud@gmail.com,pillarallison@gmail.com,sirleena@yahoo.com,annatreidler@gmail.com,allie.baxter2121@gmail.com,courtney.stull@gmail.com,lmstorm14@gmail.com,None,ekisin@gmail.com,...,camplund@comcast.net,zuinmarianna.zuin@gmail.com,hanhv93@gmail.com,orikdesign@gmail.com,tibszol@yahoo.co.uk,evelyngulbinski@yahoo.de,tibszol@yahoo.co.uk,carolinexucla@gmail.com,dagmar.ringel@gmx.de,krispolidano@hotmail.com
Add,"2687 Greenbush Rd, None, Charlotte, Vermont","20 Eastchester Rd, Apt 4A, New Rochelle, New York","2603 N Harding Ave, Unit 2, Chicago, Illinois","7985 Peak Ct., None, Riverside, California","1945 Stuart Street, Apt 4, Berkeley, California","679 West Wrightwood Avenue, 2N, Chicago, Illinois","1422 North Negley Avenue, None, Pittsburgh, Pe...","1918 Balboa Drive, None, Roseville, California","Mehringdamm 56, None, Berlin, None","23 Waverly Place, 5Y, New York, New York",...,"40 Terrace Avenue, None, San Anselmo, California","Via firenze 54, 54, Carmignano di brenta, Padova","1226 Southeast Umatilla Street, None, Portland...","6840 Balsam Way, , Oakland, California","8,Dickens Road, Harbury, Harbury, Leamington S...","Neugasse 20, , Konstanz, None","8,Dickens Road, Harbury, Harbury, Leamington S...","55 Rue Raoul Briquet, , Achicourt, None","Schulauer Moorweg 17, , Wedel, None","Pantheon Flat 1, Triq il-huttaf, Triq il-hutta..."
